# Quarterly panel imputation

This notebook loads `quarterly_panel.pkl` or `quarterly_panel.jsonl` from `ROOT/out`, then performs the requested imputations:

1. **Rent prices** and **rent contracts**  
   - linear interpolation for internal gaps  
   - forward fill / backward fill for endpoint gaps

2. **Population**  
   - annual forward imputation using a **constant growth rate based on the most recent observed annual growth**

3. **Income**  
   - annual forward imputation using a **constant growth rate based on the most recent observed annual growth**

The notebook then writes the imputed panel back to `ROOT/out`.


In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PKL_PATH = OUT_DIR / "quarterly_panel.pkl"
JSONL_PATH = OUT_DIR / "quarterly_panel.jsonl"

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("quarterly_panel.pkl exists:", PKL_PATH.exists())
print("quarterly_panel.jsonl exists:", JSONL_PATH.exists())

ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
quarterly_panel.pkl exists: True
quarterly_panel.jsonl exists: True


In [3]:
def parse_dictish(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, dict):
        return x
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        return ast.literal_eval(s)
    except Exception:
        return np.nan

if PKL_PATH.exists():
    df = pd.read_pickle(PKL_PATH)
    source_used = PKL_PATH
elif JSONL_PATH.exists():
    df = pd.read_json(JSONL_PATH, lines=True)
    for col in ["Population", "Migration", "Income"]:
        if col in df.columns:
            df[col] = df[col].map(parse_dictish)
    source_used = JSONL_PATH
else:
    raise FileNotFoundError(
        f"Neither {PKL_PATH.name} nor {JSONL_PATH.name} exists in {OUT_DIR}"
    )

df.columns = [str(c).strip() for c in df.columns]

required_cols = ["Code", "Municipality", "Year", "Quarter", "Prices", "Contracts", "Population", "Migration", "Income"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df["Code"] = df["Code"].astype(str).str.zfill(5)
df["Municipality"] = df["Municipality"].astype(str)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["Prices"] = pd.to_numeric(df["Prices"], errors="coerce")
df["Contracts"] = pd.to_numeric(df["Contracts"], errors="coerce")

quarter_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
df = df[df["Quarter"].isin(quarter_order)].copy()
df["Quarter_num"] = df["Quarter"].map(quarter_order).astype("Int64")

df = (
    df.sort_values(["Code", "Year", "Quarter_num"])
      .reset_index(drop=True)
)

print("Loaded from:", source_used)
print("Shape:", df.shape)
display(df.head(8))

Loaded from: /home/venatio/documents/university/public-sector-2026/out/quarterly_panel.pkl
Shape: (56908, 10)


,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income,Quarter_num
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN,1
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN,2
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN,3
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN,4
4,02050,NaN,2011,Q1,NaN,NaN,NaN,NaN,NaN,1
5,02050,NaN,2011,Q2,NaN,NaN,NaN,NaN,NaN,2
6,02050,NaN,2011,Q3,NaN,NaN,NaN,NaN,NaN,3
7,02050,NaN,2011,Q4,NaN,NaN,NaN,NaN,NaN,4


## Impute quarterly prices and contracts

Within each municipality:

- internal gaps are linearly interpolated
- leading and trailing endpoint gaps are filled with backward fill / forward fill


In [4]:
def impute_quarterly_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").copy()
    s = s.interpolate(method="linear", limit_direction="both")
    s = s.ffill().bfill()
    return s

df_imputed = df.copy()

for col in ["Prices", "Contracts"]:
    df_imputed[col] = (
        df_imputed.groupby("Code", group_keys=False)[col]
        .apply(impute_quarterly_series)
    )

display(df_imputed.head(8))

,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income,Quarter_num
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN,1
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN,2
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN,3
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN,4
4,02050,NaN,2011,Q1,NaN,NaN,NaN,NaN,NaN,1
5,02050,NaN,2011,Q2,NaN,NaN,NaN,NaN,NaN,2
6,02050,NaN,2011,Q3,NaN,NaN,NaN,NaN,NaN,3
7,02050,NaN,2011,Q4,NaN,NaN,NaN,NaN,NaN,4


## Build one annual row per municipality-year for Population and Income

`Population` and `Income` are annual covariates repeated across quarters, so the annual imputation is done at the municipality-year level and then merged back into the quarterly panel.


In [5]:
annual = (
    df_imputed.loc[:, ["Code", "Municipality", "Year", "Population", "Income"]]
    .drop_duplicates(subset=["Code", "Municipality", "Year"], keep="first")
    .sort_values(["Code", "Year"])
    .reset_index(drop=True)
)

print("Annual rows:", annual.shape)
display(annual.head(8))

Annual rows: (14227, 5)


,Code,Municipality,Year,Population,Income
0,00005,NaN,2015,NaN,NaN
1,02050,NaN,2011,NaN,NaN
2,02050,NaN,2012,NaN,NaN
3,02050,NaN,2013,NaN,NaN
4,02050,NaN,2014,NaN,NaN
5,08001,Abrera,2011,"{'Total': {'Total': 11611, 'Males': 5926, 'Fem...","{'Tax Base': 21049.0, 'Tax Liability': 4337.0}"
6,08001,Abrera,2012,"{'Total': {'Total': 11870, 'Males': 6072, 'Fem...","{'Tax Base': 21049.0, 'Tax Liability': 4337.0}"
7,08001,Abrera,2013,"{'Total': {'Total': 12055, 'Males': 6152, 'Fem...","{'Tax Base': 20783.0, 'Tax Liability': 4719.0}"


In [6]:
POP_PATHS = [
    ("Total", "Total"),
    ("Total", "Males"),
    ("Total", "Females"),
    ("Native", "Total"),
    ("Native", "Males"),
    ("Native", "Females"),
    ("Foreign", "Total"),
    ("Foreign", "Males"),
    ("Foreign", "Females"),
]

INC_PATHS = [
    ("Tax Base",),
    ("Tax Liability",),
]

def get_nested_value(d, path):
    if not isinstance(d, dict):
        return np.nan
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return np.nan
        cur = cur[key]
    return cur

def flatten_population_dict(d):
    out = {}
    for g1, g2 in POP_PATHS:
        out[f"pop__{g1}__{g2}"] = pd.to_numeric(get_nested_value(d, (g1, g2)), errors="coerce")
    return out

def flatten_income_dict(d):
    out = {}
    for (k,) in INC_PATHS:
        out[f"inc__{k.replace(' ', '_')}"] = pd.to_numeric(get_nested_value(d, (k,)), errors="coerce")
    return out

pop_flat = annual["Population"].map(flatten_population_dict).apply(pd.Series)
inc_flat = annual["Income"].map(flatten_income_dict).apply(pd.Series)

annual_flat = pd.concat(
    [annual[["Code", "Municipality", "Year"]], pop_flat, inc_flat],
    axis=1,
)

display(annual_flat.head(5))

,Code,Municipality,Year,pop__Total__Total,pop__Total__Males,pop__Total__Females,pop__Native__Total,pop__Native__Males,pop__Native__Females,pop__Foreign__Total,pop__Foreign__Males,pop__Foreign__Females,inc__Tax_Base,inc__Tax_Liability
0,00005,NaN,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02050,NaN,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02050,NaN,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02050,NaN,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Forward impute Population and Income using constant previous growth

For each municipality and each annual component:

- compute the **most recent observed annual growth rate**
- if the last observed years are not adjacent, annualize the growth across the gap
- project that same rate forward into missing future years
- if there is not enough history to estimate growth, use `0.0` growth


In [7]:
def estimate_last_annual_growth(series: pd.Series, years: pd.Series) -> float:
    tmp = pd.DataFrame({
        "year": pd.to_numeric(years, errors="coerce"),
        "value": pd.to_numeric(series, errors="coerce"),
    }).dropna()

    tmp = tmp.sort_values("year").reset_index(drop=True)

    if len(tmp) < 2:
        return 0.0

    last = tmp.iloc[-1]
    prev = tmp.iloc[-2]

    prev_val = float(prev["value"])
    last_val = float(last["value"])
    year_gap = int(last["year"] - prev["year"])

    if year_gap <= 0 or prev_val <= 0 or last_val <= 0:
        return 0.0

    return (last_val / prev_val) ** (1 / year_gap) - 1

def impute_forward_constant_growth(series: pd.Series, years: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").copy()
    y = pd.to_numeric(years, errors="coerce").astype("Int64")

    obs_mask = s.notna()
    if obs_mask.sum() == 0:
        return s

    rate = estimate_last_annual_growth(s, y)

    observed = pd.DataFrame({"year": y, "value": s}).dropna().sort_values("year")
    last_obs_year = int(observed.iloc[-1]["year"])
    last_obs_value = float(observed.iloc[-1]["value"])

    future_years = sorted(y[(s.isna()) & (y > last_obs_year)].dropna().astype(int).unique())

    prev_year = last_obs_year
    prev_value = last_obs_value

    for yr in future_years:
        gap = yr - prev_year
        next_value = prev_value * ((1 + rate) ** gap)
        s.loc[y == yr] = next_value
        prev_year = yr
        prev_value = next_value

    return s

In [8]:
annual_imputed = annual_flat.copy()

pop_cols = [c for c in annual_imputed.columns if c.startswith("pop__")]
inc_cols = [c for c in annual_imputed.columns if c.startswith("inc__")]

for col in pop_cols + inc_cols:
    annual_imputed[col] = (
        annual_imputed.groupby("Code", group_keys=False)
        .apply(lambda g: impute_forward_constant_growth(g[col], g["Year"]))
        .reset_index(level=0, drop=True)
    )

display(annual_imputed.head(5))

,Code,Municipality,Year,pop__Total__Total,pop__Total__Males,pop__Total__Females,pop__Native__Total,pop__Native__Males,pop__Native__Females,pop__Foreign__Total,pop__Foreign__Males,pop__Foreign__Females,inc__Tax_Base,inc__Tax_Liability
0,00005,NaN,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02050,NaN,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02050,NaN,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02050,NaN,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
def int_or_none(x):
    return None if pd.isna(x) else int(round(float(x)))

def float_or_none(x):
    return None if pd.isna(x) else float(x)

def row_to_population_dict(row):
    return {
        "Total": {
            "Total": int_or_none(row["pop__Total__Total"]),
            "Males": int_or_none(row["pop__Total__Males"]),
            "Females": int_or_none(row["pop__Total__Females"]),
        },
        "Native": {
            "Total": int_or_none(row["pop__Native__Total"]),
            "Males": int_or_none(row["pop__Native__Males"]),
            "Females": int_or_none(row["pop__Native__Females"]),
        },
        "Foreign": {
            "Total": int_or_none(row["pop__Foreign__Total"]),
            "Males": int_or_none(row["pop__Foreign__Males"]),
            "Females": int_or_none(row["pop__Foreign__Females"]),
        },
    }

def row_to_income_dict(row):
    return {
        "Tax Base": float_or_none(row["inc__Tax_Base"]),
        "Tax Liability": float_or_none(row["inc__Tax_Liability"]),
    }

annual_imputed["Population"] = annual_imputed.apply(row_to_population_dict, axis=1)
annual_imputed["Income"] = annual_imputed.apply(row_to_income_dict, axis=1)

annual_small = annual_imputed[["Code", "Year", "Population", "Income"]].copy()
display(annual_small.head(5))

,Code,Year,Population,Income
0,00005,2015,"{'Total': {'Total': None, 'Males': None, 'Fema...","{'Tax Base': None, 'Tax Liability': None}"
1,02050,2011,"{'Total': {'Total': None, 'Males': None, 'Fema...","{'Tax Base': None, 'Tax Liability': None}"
2,02050,2012,"{'Total': {'Total': None, 'Males': None, 'Fema...","{'Tax Base': None, 'Tax Liability': None}"
3,02050,2013,"{'Total': {'Total': None, 'Males': None, 'Fema...","{'Tax Base': None, 'Tax Liability': None}"
4,02050,2014,"{'Total': {'Total': None, 'Males': None, 'Fema...","{'Tax Base': None, 'Tax Liability': None}"


## Merge imputed annual covariates back into the quarterly panel


In [10]:
df_final = (
    df_imputed.drop(columns=["Population", "Income"], errors="ignore")
    .merge(annual_small, on=["Code", "Year"], how="left")
    .sort_values(["Code", "Year", "Quarter_num"])
    .reset_index(drop=True)
)

df_final = df_final[
    ["Code", "Municipality", "Year", "Quarter", "Prices", "Contracts", "Population", "Migration", "Income"]
].copy()

print("Final shape:", df_final.shape)
display(df_final.head(8))

Final shape: (56908, 9)


,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income
0,00005,NaN,2015,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
1,00005,NaN,2015,Q2,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
2,00005,NaN,2015,Q3,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
3,00005,NaN,2015,Q4,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
4,02050,NaN,2011,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
5,02050,NaN,2011,Q2,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
6,02050,NaN,2011,Q3,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"
7,02050,NaN,2011,Q4,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}"


## Optional diagnostics


In [11]:
print("Missing Prices:", df_final["Prices"].isna().sum())
print("Missing Contracts:", df_final["Contracts"].isna().sum())
print("Missing Population:", df_final["Population"].isna().sum())
print("Missing Income:", df_final["Income"].isna().sum())
print("Missing Migration:", df_final["Migration"].isna().sum())

Missing Prices: 80
Missing Contracts: 80
Missing Population: 0
Missing Income: 0
Missing Migration: 15240


## Save outputs


In [12]:
IMPUTED_PKL_PATH = OUT_DIR / "quarterly_panel_imputed.pkl"
IMPUTED_JSONL_PATH = OUT_DIR / "quarterly_panel_imputed.jsonl"
IMPUTED_CSV_PATH = OUT_DIR / "quarterly_panel_imputed.csv"

df_final.to_pickle(IMPUTED_PKL_PATH)
df_final.to_json(IMPUTED_JSONL_PATH, orient="records", lines=True, force_ascii=False)

# CSV is optional and mainly for inspection; dict columns will be stringified there
df_final.to_csv(IMPUTED_CSV_PATH, index=False)

print("Saved:")
print(" -", IMPUTED_PKL_PATH)
print(" -", IMPUTED_JSONL_PATH)
print(" -", IMPUTED_CSV_PATH)

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_imputed.pkl
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_imputed.jsonl
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_imputed.csv
